# Checks and examples, with and without an ecoinvent licence

**English** | [Français](check_and_examples.fr.ipynb)

This notebook gathers the checks and examples run on the repository:

- **Part 1, with an ecoinvent licence**: import ecoinvent 3.11 cut-off into Brightway, recompute the fabrication impacts and check that they are the same as the shipped ones (computed with Activity Browser) in the 16 EF 3.1 categories; then an example of an inventory change.
- **Part 2, without a licence**: the reference results and three changes to the model.

The expected values are given above each cell. Run the cells in order. In the commands, `!` runs a terminal command and `{sys.executable}` is the Python of the notebook.

## 0. Download the repository and install (once)

Without Git: the repository is downloaded as a ZIP into your home folder. The second `pip` line is only needed for Part 1.

In [ ]:
import sys, os, urllib.request, zipfile
os.chdir(os.path.expanduser("~"))
urllib.request.urlretrieve("https://github.com/Ayabh12/h2-ageing-lca/archive/refs/heads/main.zip", "h2-ageing-lca.zip")
zipfile.ZipFile("h2-ageing-lca.zip").extractall(".")
os.chdir("h2-ageing-lca-main")
!{sys.executable} -m pip install -r requirements.txt
!{sys.executable} -m pip install -U -r requirements-brightway.txt

After installing Brightway, **restart the kernel** (menu *Kernel → Restart*), then go on.

# Part 1 – Check of the fabrication calculation with ecoinvent

Needs an ecoinvent licence (your own or your institution's, if its terms allow it). The login is never written to a file: do not type it in a cell you share.

### 1.1 ecoinvent login

The password is not shown while you type.

In [ ]:
import sys, os
import pandas as pd
os.chdir(os.path.expanduser("~/h2-ageing-lca-main"))
import getpass
os.environ["EI_USERNAME"] = input("ecoinvent username: ")
os.environ["EI_PASSWORD"] = getpass.getpass("ecoinvent password: ")

### 1.2 Import ecoinvent and recompute the fabrication impacts

The first time, ecoinvent 3.11 cut-off is downloaded and imported into the Brightway project `h2ageing` (from about 20 minutes to under an hour). The next times, the calculation takes about one minute.

**Expected**: the list of the 16 EF 3.1 methods used, then the climate change impact of each component: `fuel_cell_stack_128kW` 3698.3, `electrolyser_stack_1p25MW` 39509.4, `tank_type_IV` 2332.0 kg CO2-eq.

In [ ]:
!{sys.executable} scripts/build_fabrication_brightway.py --project h2ageing --import-ecoinvent --out results/fabrication_test.csv

### 1.3 Check of the 16 EF 3.1 categories

Difference (in %) between the impacts recomputed with Brightway and the Activity Browser impacts shipped in the repository.

**Expected**: 0 everywhere (−0 is a rounding); difference below 0.01 % in the 16 categories and for the three components.

In [ ]:
new = pd.read_csv("results/fabrication_test.csv").set_index("category")
shipped = pd.read_csv("data/impacts/fabrication_per_unit.csv").set_index("category")
diff = ((new - shipped) / shipped * 100).round(2)
print("Largest difference:", diff.abs().max().max(), "%")
diff

### 1.4 Example: manufacture the fuel cell stack with Japanese electricity

The Mirai stack is manufactured by Toyota in Japan. The French residual mix is replaced by the Japanese high-voltage electricity market (ecoinvent 3.11) for the 1 096 kWh used to manufacture the stack, then the whole model is run again. The original inventory is restored at the end of the cell.

**Expected**: about 4 316 kg CO2-eq per fuel cell stack instead of 3 698; chain (g CO2-eq/km): no ageing 74.1 → 77.2; central 97.2 → 103.3; rise due to ageing in the central case +31.1 % → +33.8 %.

In [ ]:
p = "data/inventories/fuel_cell_stack_128kW.csv"
original = open(p, encoding="utf-8", newline="").read()
inv = pd.read_csv(p, dtype=str, keep_default_na=False)
row = inv["input"].str.contains("French residual mix")
inv.loc[row, "input"] = "Electricity, Japan, high voltage"
inv.loc[row, "ecoinvent_activity"] = "market for electricity, high voltage"
inv.loc[row, "location"] = "JP"
inv.to_csv(p, index=False)
try:
    !{sys.executable} scripts/build_fabrication_brightway.py --project h2ageing --out results/fabrication_japan.csv
finally:
    open(p, "w", encoding="utf-8", newline="").write(original)   # original inventory restored

!{sys.executable} scripts/run.py --out results/reference > {os.devnull}
!{sys.executable} scripts/run.py --set fabrication.file=results/fabrication_japan.csv --out results/japan > {os.devnull}
ref = pd.read_csv("results/reference/chain.csv").set_index("case")
jp = pd.read_csv("results/japan/chain.csv").set_index("case")
pd.DataFrame({"France: total (g/km)": ref["total_g_per_km"], "Japan: total (g/km)": jp["total_g_per_km"],
              "France: rise (%)": ref["rise_pct"], "Japan: rise (%)": jp["rise_pct"]}).round(1)

# Part 2 – Use and change the model without ecoinvent

The model uses the fabrication impacts shipped in the repository (`data/impacts`). No licence is needed. These results are also checked automatically by `tests/test_reference_results.py`.

In [ ]:
import sys, os
import pandas as pd
os.chdir(os.path.expanduser("~/h2-ageing-lca-main"))

## Example 1 – Reference results

**Expected** (g CO2-eq/km): no ageing 74.1; optimistic 77.8 (+5.0 %); central 97.2 (+31.1 %); pessimistic 100.1 (+35.0 %).

In [ ]:
!{sys.executable} scripts/run.py --out results/reference > {os.devnull}
ref = pd.read_csv("results/reference/chain.csv").set_index("case")
ref[["total_g_per_km", "rise_pct"]].round(1)

## Example 2 – Change the fuel cell lifetime: 6 000 h instead of 4 100 h

**Expected**, central case: 1 fuel cell stack instead of 2; chain 97.2 → 78.9 g/km; rise +31.1 % → +6.4 %. Above 5 554 h, one stack lasts the whole vehicle life.

In [ ]:
!{sys.executable} scripts/run.py --set cases.central.fuel_cell.hours_to_end_of_life=6000 --out results/pile_6000h > {os.devnull}
life = pd.read_csv("results/pile_6000h/life.csv").set_index("case")
p6 = pd.read_csv("results/pile_6000h/chain.csv").set_index("case")
display(life[["fuel_cell_end_of_life_h", "fuel_cell_stacks"]].round(0))
pd.DataFrame({"Reference (g/km)": ref["total_g_per_km"], "Fuel cell 6 000 h (g/km)": p6["total_g_per_km"],
              "Reference rise (%)": ref["rise_pct"], "6 000 h rise (%)": p6["rise_pct"]}).round(1)

## Example 3 – Wind electricity for the electrolyser

**Expected**: no ageing 74.1 → 41.2 g/km; central 97.2 → 61.3 g/km; rise due to ageing +31.1 % → +48.6 %. With clean electricity, the replacement stacks weigh more in proportion. Other choices: `average_mix_FR`, `nuclear_FR`, `photovoltaic`.

In [ ]:
!{sys.executable} scripts/run.py --set usage.electricity_electrolyser=wind_FR --out results/eolien > {os.devnull}
eo = pd.read_csv("results/eolien/chain.csv").set_index("case")
pd.DataFrame({"Residual mix (g/km)": ref["total_g_per_km"], "Wind (g/km)": eo["total_g_per_km"],
              "Residual mix rise (%)": ref["rise_pct"], "Wind rise (%)": eo["rise_pct"]}).round(1)

## Example 4 – Use your own fabrication impact

Test value: fuel cell stack 30 % lower. With a value from another study, replace `*= 0.7` by `= <value>` (kg CO2-eq per stack).

**Expected**: no ageing 74.1 → 68.6 g/km; central 97.2 → 86.1 g/km; rise +31.1 % → +25.5 %.

In [ ]:
fab = pd.read_csv("data/impacts/fabrication_per_unit.csv").set_index("category")
fab.loc["climate_change", "fuel_cell_stack_128kW"] *= 0.7
os.makedirs("results", exist_ok=True)
fab.to_csv("results/my_fabrication.csv")
!{sys.executable} scripts/run.py --set fabrication.file=results/my_fabrication.csv --out results/my_fabrication > {os.devnull}
mf = pd.read_csv("results/my_fabrication/chain.csv").set_index("case")
pd.DataFrame({"Reference (g/km)": ref["total_g_per_km"], "Stack −30 % (g/km)": mf["total_g_per_km"],
              "Reference rise (%)": ref["rise_pct"], "Stack −30 % rise (%)": mf["rise_pct"]}).round(1)